# Emulating Comoving Angular Distance with tissage-cosmique

This notebook demonstrates the full workflow:
1. Run `comoving_angular_distance` over a grid of cosmological parameters
2. Train a Gaussian Process emulator on the results
3. Validate accuracy against pyccl truth
4. Compare predictions and residuals visually

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tissage_cosmique.computations.distances import comoving_angular_distance
from tissage_cosmique.emulators import (
    GPEmulator,
    build_training_data,
    params_to_feature_matrix,
    validate_against_computation,
    check_calibration,
)

## 1. Define the parameter space and scale factor grid

In [ ]:
PARAM_NAMES = ["Omega_c", "h", "sigma8"]
FIXED_PARAMS = dict(Omega_b=0.0486, n_s=0.9667, Omega_k=0.0, w0=-1.0, wa=0.0)

# Scale factors from z~4 (a=0.2) to z~0.05 (a=0.95)
a_grid = np.linspace(0.2, 0.8, 30)

# Generate random cosmologies within a plausible range
rng = np.random.default_rng(42)
n_train = 50
n_test = 10

def make_samples(n, rng):
    return [
        {
            "Omega_c": rng.uniform(0.22, 0.32),
            "h": rng.uniform(0.62, 0.75),
            "sigma8": rng.uniform(0.77, 0.87),
            **FIXED_PARAMS,
        }
        for _ in range(n)
    ]

train_samples = make_samples(n_train, rng)
test_samples = make_samples(n_test, rng)

print(f"Training: {n_train} cosmologies x {len(a_grid)} scale factors = {n_train * len(a_grid)} points")
print(f"Test:     {n_test} cosmologies x {len(a_grid)} scale factors = {n_test * len(a_grid)} points")

## 2. Generate training data by running pyccl

In [ ]:
%%time
X_train, y_train = build_training_data(
    comoving_angular_distance, train_samples, a_grid, param_names=PARAM_NAMES
)
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"Distance range: [{y_train.min():.1f}, {y_train.max():.1f}] Mpc")

## 3. Train the GP emulator

In [ ]:
%%time
emu = GPEmulator(feature_names=PARAM_NAMES + ["a"])
emu.fit(X_train, y_train)

meta = emu.metadata
print(f"Fitted: {meta['is_fitted']}")
print(f"Training samples: {meta['n_training_samples']}")
print(f"Training score (R2): {meta['training_score']:.6f}")
print(f"Optimized kernel: {meta['kernel']}")

## 4. Validate against pyccl on held-out cosmologies

In [ ]:
result = validate_against_computation(
    emu, comoving_angular_distance, test_samples, a_grid, param_names=PARAM_NAMES
)

print(f"Test set size:         {result.n_test}")
print(f"R2 score:              {result.r2_score:.6f}")
print(f"MAE:                   {result.mae:.2f} Mpc")
print(f"RMSE:                  {result.rmse:.2f} Mpc")
print(f"Max absolute error:    {result.max_abs_error:.2f} Mpc")
print(f"Mean relative error:   {result.mean_relative_error:.4%}")
print(f"Max relative error:    {result.max_relative_error:.4%}")
print(f"95th pctl abs error:   {result.percentile_95_error:.2f} Mpc")

## 5. Visual comparison: emulator vs pyccl for a few test cosmologies

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, params in enumerate(test_samples[:3]):
    truth = comoving_angular_distance(params, a_grid)
    X_pred = params_to_feature_matrix(params, a_grid, param_names=PARAM_NAMES)
    pred, std = emu.predict_with_std(X_pred)

    ax = axes[i]
    ax.plot(a_grid, truth, "k-", lw=2, label="pyccl")
    ax.plot(a_grid, pred, "r--", lw=1.5, label="GP emulator")
    ax.fill_between(a_grid, pred - 2 * std, pred + 2 * std, alpha=0.2, color="red", label=r"$\pm 2\sigma$")
    ax.set_xlabel("Scale factor a")
    ax.set_ylabel("Comoving angular distance [Mpc]")
    label = f"$\\Omega_c$={params['Omega_c']:.3f}, h={params['h']:.3f}"
    ax.set_title(label, fontsize=10)
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle("Emulator vs pyccl — held-out cosmologies", fontsize=12)
plt.tight_layout()
plt.show()

## 6. Residuals distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Absolute residuals
axes[0].hist(result.residuals, bins=40, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_xlabel("Residual (pred - truth) [Mpc]")
axes[0].set_ylabel("Count")
axes[0].set_title("Absolute residuals")

# Relative errors per scale factor
X_test, y_test = build_training_data(comoving_angular_distance, test_samples, a_grid, param_names=PARAM_NAMES)
y_pred = emu.predict(X_test)
rel_err = np.abs(y_pred - y_test) / np.maximum(np.abs(y_test), 1.0)
rel_err_by_a = rel_err.reshape(n_test, len(a_grid))

axes[1].plot(a_grid, rel_err_by_a.T, alpha=0.3, color="steelblue")
axes[1].plot(a_grid, np.mean(rel_err_by_a, axis=0), "k-", lw=2, label="mean")
axes[1].set_xlabel("Scale factor a")
axes[1].set_ylabel("Relative error |pred - truth| / |truth|")
axes[1].set_title("Relative error vs scale factor")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Uncertainty calibration check

In [ ]:
cal = check_calibration(emu, X_test, y_test)

print("Coverage calibration:")
for exp, obs in zip(cal.expected_coverage, cal.observed_coverage):
    print(f"  Expected {exp:.0%} -> Observed {obs:.1%}")
print(f"\nMean predicted std:           {cal.mean_std:.2f} Mpc")
print(f"Std vs error correlation:     {cal.std_vs_error_correlation:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax.plot(cal.expected_coverage, cal.observed_coverage, "ro-", markersize=8, label="GP emulator")
ax.set_xlabel("Expected coverage")
ax.set_ylabel("Observed coverage")
ax.set_title("Uncertainty calibration")
ax.legend()
ax.set_xlim(0.5, 1.05)
ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.show()

### Why is the GP conservative?

The observed coverage is higher than expected at every level (e.g., 99% observed at the
68% level). This means the GP's predicted uncertainties are **too wide** — it is
*overconfident in its own uncertainty*, or equivalently, *conservative* in its predictions.

This is typical for GP emulation of **noise-free, deterministic functions** like pyccl
computations. Several factors contribute:

1. **No observation noise.** The training data has zero noise, but the GP still includes a
   small regularization term (`alpha=1e-6`) to keep the kernel matrix well-conditioned.
   This injects a fictitious noise floor that inflates the predictive variance slightly.

2. **Smooth target function.** Comoving angular distance is a very smooth function of the
   cosmological parameters. The RBF kernel captures this well, so prediction errors are
   tiny (MAE ~ 0.4 Mpc on a ~1000–8000 Mpc range), but the GP's variance estimate
   doesn't shrink as aggressively because it is set by the kernel length scale, not by the
   actual residuals.

3. **Dense training relative to parameter volume.** With 50 cosmologies in a compact
   3-parameter space, most test points are well-interpolated. The GP "knows" it is
   interpolating but still assigns uncertainty based on distance to training points in the
   kernel metric, which overestimates the true error.

**In practice this is the safe direction** — a conservative emulator won't silently produce
wrong answers. If tighter uncertainty bands are needed (e.g., for MCMC sampling), options
include:
- Increasing training set size (reduces the gap between predicted and actual error)
- Post-hoc recalibration (scale the predicted std to match observed coverage on a validation set)
- Using a different kernel or fitting the noise level as a free parameter

## 8. Save and reload the emulator

In [ ]:
emu.save("distances_gp_emulator.joblib")
loaded = GPEmulator.load("distances_gp_emulator.joblib")

# Verify roundtrip
params = test_samples[0]
X_check = params_to_feature_matrix(params, a_grid, param_names=PARAM_NAMES)
np.testing.assert_allclose(emu.predict(X_check), loaded.predict(X_check))
print("Save/load roundtrip: OK")
print(f"Loaded metadata: {loaded.metadata}")